# NLP Masterclass 02: Word Embeddings & Word2Vec

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** NLP 01 (Tokenization), ML Fundamentals  
**Difficulty:** ⭐⭐ Intermediate  
**Estimated Time:** 90–120 minutes

---

## 🎓 Socratic Deep Check

> *"Word2Vec showed that king - man + woman ≈ queen. These embeddings capture semantic meaning as geometry. But Word2Vec gives 'bank' the SAME vector whether it means 'river bank' or 'financial bank'. How do contextual embeddings (BERT, GPT) solve this fundamental limitation?"*

---

## Why Embeddings Are the Foundation of Modern AI

Computers understand numbers, not words. **Embeddings convert words into dense vectors** where similar words live close together. This single idea powers:
- **RAG** (NB28): Embed documents, find similar ones
- **Vector databases** (NB27): Store and search embeddings
- **LLM internals**: Every token in GPT starts as an embedding

## Table of Contents
1. From One-Hot to Dense Vectors
2. Word2Vec: Skip-Gram from Scratch
3. Vector Arithmetic & Analogies
4. Pre-trained Embeddings (GloVe, FastText)
5. From Static to Contextual Embeddings

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors use embeddings as magic black boxes. Seniors understand that embedding quality determines every downstream task's ceiling — bad embeddings = bad retrieval = bad RAG = bad LLM answers. Choosing between OpenAI embeddings, sentence-transformers, and domain-specific embeddings is a critical production decision.

**Mental Model:** A word embedding is a GPS coordinate in meaning-space. "King" and "Queen" are close because they share royalty-ness. "King" and "Man" share a gender vector. Subtracting gender and adding the opposite gender moves you: King - Man + Woman = Queen.

**Common Junior Pitfall:** Using general-purpose embeddings for domain-specific tasks. Medical text needs medical embeddings. Legal text needs legal embeddings. Fine-tuned embeddings can improve RAG recall by 20-40%.

---

In [ ]:
!pip install -q numpy matplotlib scikit-learn gensim

## 1 · From One-Hot to Dense Vectors

### The Problem with One-Hot Encoding

```
Vocabulary: [cat, dog, fish, king, queen]

One-hot 'cat':   [1, 0, 0, 0, 0]  ← All words equidistant!
One-hot 'dog':   [0, 1, 0, 0, 0]  ← cos(cat, dog) = 0
One-hot 'fish':  [0, 0, 1, 0, 0]  ← cos(cat, fish) = 0 
```

One-hot says cat and dog are as different as cat and king. This is useless for NLP.

### Dense Embeddings Fix This

```
Embedding 'cat':   [0.2, 0.8, -0.1, 0.5]  ← Similar to dog!
Embedding 'dog':   [0.3, 0.7, -0.2, 0.4]  ← cos(cat, dog) = 0.98
Embedding 'king':  [0.9, -0.3, 0.7, 0.1]  ← cos(cat, king) = 0.12
```

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Simulate meaningful embeddings
embeddings = {
    'king':   np.array([0.9, 0.1, 0.8, 0.2]),
    'queen':  np.array([0.9, 0.1, 0.2, 0.8]),
    'man':    np.array([0.1, 0.1, 0.8, 0.2]),
    'woman':  np.array([0.1, 0.1, 0.2, 0.8]),
    'cat':    np.array([0.2, 0.9, 0.5, 0.5]),
    'dog':    np.array([0.3, 0.85, 0.5, 0.5]),
}

# Cosine similarity
print('Cosine Similarities:')
pairs = [('king','queen'), ('king','man'), ('cat','dog'), ('cat','king')]
for w1, w2 in pairs:
    sim = cosine_similarity([embeddings[w1]], [embeddings[w2]])[0][0]
    print(f'  {w1:>5} ↔ {w2:<5}: {sim:.3f}')

# THE FAMOUS ANALOGY: king - man + woman ≈ queen
result = embeddings['king'] - embeddings['man'] + embeddings['woman']
print(f'\n"king" - "man" + "woman" = ?')
best_word, best_sim = None, -1
for word, vec in embeddings.items():
    if word in ['king', 'man', 'woman']:
        continue
    sim = cosine_similarity([result], [vec])[0][0]
    print(f'  Similarity to "{word}": {sim:.3f}')
    if sim > best_sim:
        best_word, best_sim = word, sim
print(f'\n  Answer: "{best_word}" (similarity={best_sim:.3f}) ✓')

---
## 2 · Word2Vec: Skip-Gram from Scratch

### The Core Idea

**"You shall know a word by the company it keeps."** — J.R. Firth

| Model | Task | Example |
|-------|------|---------|
| **CBOW** | Predict center from context | [The, _, sat] → ? = "cat" |
| **Skip-Gram** | Predict context from center | "cat" → ? = [The, sat] |

Skip-Gram works better for rare words and is used more in practice.

In [ ]:
import numpy as np

class SkipGram:
    """Skip-Gram Word2Vec from scratch using NumPy.
    
    Two matrices:
      W_in  (vocab_size x embed_dim): Input embeddings (THESE are the word vectors we want)
      W_out (embed_dim x vocab_size): Output embeddings (used only during training)
    """
    
    def __init__(self, vocab_size, embed_dim):
        self.embed_dim = embed_dim
        # Xavier initialization
        self.W_in = np.random.randn(vocab_size, embed_dim) * 0.01
        self.W_out = np.random.randn(embed_dim, vocab_size) * 0.01
    
    def forward(self, center_idx):
        """Look up embedding for center word."""
        h = self.W_in[center_idx]  # Hidden layer (the embedding)
        scores = h @ self.W_out    # Score every word in vocab
        # Softmax
        exp_scores = np.exp(scores - np.max(scores))
        probs = exp_scores / exp_scores.sum()
        return h, probs
    
    def train_step(self, center_idx, context_idx, lr=0.01):
        """One gradient descent step."""
        h, probs = self.forward(center_idx)
        
        # Loss = -log(P(context|center))
        loss = -np.log(probs[context_idx] + 1e-15)
        
        # Gradient for W_out
        dscores = probs.copy()
        dscores[context_idx] -= 1  # Subtract 1 from correct context
        dW_out = np.outer(h, dscores)
        
        # Gradient for W_in
        dh = self.W_out @ dscores
        
        # Update
        self.W_out -= lr * dW_out
        self.W_in[center_idx] -= lr * dh
        
        return loss

# Train on a tiny corpus
corpus = "the cat sat on the mat the cat ate the food"
words = corpus.split()
vocab = sorted(set(words))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

model = SkipGram(len(vocab), embed_dim=8)
window_size = 2

# Generate training pairs
for epoch in range(200):
    total_loss = 0
    for i, word in enumerate(words):
        center = word2idx[word]
        for j in range(max(0, i-window_size), min(len(words), i+window_size+1)):
            if j != i:
                context = word2idx[words[j]]
                total_loss += model.train_step(center, context)
    if epoch % 50 == 0:
        print(f'Epoch {epoch:3d}: avg loss = {total_loss/len(words):.3f}')

# Check learned similarities
print('\nLearned Word Vectors (8-dimensional):')
for i, word in enumerate(vocab):
    vec = model.W_in[i]
    print(f'  {word:>5}: [{" ".join(f"{v:+.2f}" for v in vec[:4])} ...]')

---
## 3 · Pre-trained Embeddings in Practice

In [ ]:
# Using Gensim's pre-trained Word2Vec
import gensim.downloader as api

# Load pre-trained GloVe (trained on 6B tokens from Wikipedia + Gigaword)
print('Loading GloVe embeddings (this may take a minute)...')
try:
    glove = api.load('glove-wiki-gigaword-50')  # 50-dimensional vectors

    # Famous analogy
    result = glove.most_similar(positive=['king', 'woman'], negative=['man'], topn=3)
    print('\nking - man + woman = ?')
    for word, score in result:
        print(f'  {word}: {score:.4f}')

    # Domain-specific analogies
    print('\npython - programming + cooking = ?')
    result2 = glove.most_similar(positive=['python', 'cooking'], negative=['programming'], topn=3)
    for word, score in result2:
        print(f'  {word}: {score:.4f}')

    # Similarity
    pairs = [('cat','dog'), ('cat','car'), ('good','great'), ('good','terrible')]
    print('\nSimilarities:')
    for w1, w2 in pairs:
        print(f'  {w1:>8} ↔ {w2:<8}: {glove.similarity(w1, w2):.3f}')
except Exception as e:
    print(f'Note: Could not load GloVe ({e}). This is normal in offline environments.')
    print('In production, you would use sentence-transformers or OpenAI embeddings (NB27).')

---
## 4 · From Static to Contextual Embeddings

### The Polysemy Problem

Word2Vec gives ONE vector per word. But "bank" has multiple meanings:
- "I went to the **bank** to deposit money" (financial)
- "I sat by the river **bank**" (geography)

| Embedding Type | Same word, different context | Models |
|---------------|----------------------------|--------|
| **Static** (Word2Vec/GloVe) | Same vector always | Word2Vec, GloVe, FastText |
| **Contextual** (Transformers) | Different vector per context | BERT, GPT, Sentence-BERT |

### 🎓 DEEP QUESTION ANSWERED

**Q:** *How do contextual embeddings solve polysemy?*

**A:** In transformers, each token's output embedding is the result of self-attention over ALL other tokens. When "bank" appears near "money" and "deposit", attention focuses on financial context words, producing a financial-meaning vector. When near "river" and "water", attention produces a geographical-meaning vector. **The embedding IS the context.**

This is exactly what makes sentence-transformers (NB27) and BERT embeddings so powerful for RAG (NB28).

---

## ✅ Knowledge Check

### Q1: Cosine vs Euclidean
Why do we use cosine similarity instead of Euclidean distance for comparing embeddings?

<details><summary>👉 Answer</summary>

Cosine similarity measures the angle between vectors (direction), while Euclidean measures magnitude. Two documents about the same topic might have different lengths (different magnitude vectors) but point in the same direction. Cosine similarity correctly identifies them as similar. This is why vector databases (NB27) default to cosine similarity.
</details>

### Q2: Embedding dimensions
OpenAI's text-embedding-3-large uses 3072 dimensions. Why not 10 dimensions? Why not 100,000?

<details><summary>👉 Answer</summary>

Too few dimensions (10): can't represent enough semantic distinctions. Too many (100K): wastes storage, slows search, and leads to the curse of dimensionality where distances become meaningless. 768-3072 is the sweet spot found empirically — enough capacity for rich semantics without computational waste.
</details>

### Q3: Production decision
You're building a RAG system for legal documents. Should you use general-purpose OpenAI embeddings or fine-tune a sentence-transformer on legal text?

<details><summary>👉 Answer</summary>

Fine-tune on legal text. General embeddings treat "consideration" (legal: payment/exchange in a contract) and "consideration" (common: thinking about something) the same. Domain-specific fine-tuning on legal corpora can improve retrieval recall by 20-40%, which directly improves RAG answer quality (NB28).
</details>

---

## 🔨 Practical Practice

### Exercise 1: Bias Detection
Using GloVe, compute the cosine similarity of "doctor" and "nurse" with "man" and "woman". What bias do you observe? How might this affect an AI system?

### Exercise 2: Custom Word2Vec
Extend the SkipGram class with Negative Sampling. Instead of softmax over the full vocab, sample 5 negative words per positive pair. Compare training speed.

### Exercise 3: Embedding Visualization
Use t-SNE to reduce GloVe embeddings of 50 words (mix of animals, countries, colors, professions) to 2D and plot them. Do semantically similar words cluster?

---

**Next →** NLP 03: Recurrent Networks & LSTMs